# 06 — Demo: the alert fires on corrupted data

For the presentation. Shows that the Finance validation is not just green on clean data, but actually catches a broken order.

1. Remember the current Delta version of `silver.orders`.
2. **Corrupt one order:** add $100 to its `o_totalprice` (still positive, so no `CHECK` constraint stops it; only the reconciliation rule can catch it).
3. Run `04_validation` (order-total rules only, so it takes seconds): V1.1 fails, the order shows up in `mismatched_orders_latest`, and the **alert fires** (a new mismatched order since the previous run).
4. **Restore** the table with Delta time travel (`RESTORE TABLE … TO VERSION AS OF`) and run the validation again: the mismatch is resolved.

The restore runs in a `finally` block, so silver is put back even if a step fails. The demo runs are stored with `run_label = 'demo'` so they are easy to tell apart in the monitoring history.

Prerequisites: `04_validation` has run at least once (it provides the baseline the alert compares against).

In [ ]:
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("demo_order_key", "", "Order to corrupt (empty = first order)")
dbutils.widgets.text("corruption_amount", "100.00", "Amount added to o_totalprice")

CATALOG = dbutils.widgets.get("catalog").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
MON_SCHEMA = dbutils.widgets.get("monitoring_schema").strip()
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
MON = f"{CATALOG}.{MON_SCHEMA}"
ORDERS = f"{SILVER}.orders"

import json
from decimal import Decimal

from pyspark.sql import functions as F

AMOUNT = Decimal(dbutils.widgets.get("corruption_amount").strip())
key_widget = dbutils.widgets.get("demo_order_key").strip()
ORDER_KEY = int(key_widget) if key_widget else spark.table(ORDERS).agg(F.min("o_orderkey")).first()[0]

assert spark.catalog.tableExists(f"{MON}.metric_mismatched_orders"), "Run 04_validation once before the demo."

VALIDATION_ARGS = {
    "catalog": CATALOG,
    "silver_schema": SILVER_SCHEMA,
    "monitoring_schema": MON_SCHEMA,
    "run_label": "demo",
    "rule_groups": "order_total",
    "fail_on_error": "false",   # we want the summary back, not an exception
}


def run_validation():
    return json.loads(dbutils.notebook.run("./04_validation", 3600, VALIDATION_ARGS))


def show(step, s):
    flag = "🚨 ALERT" if s.get("alert") else "no alert"
    print(f"{step}: mismatched orders = {s['mismatched_orders']:,}, new = {s['new_mismatched_orders']:,}, "
          f"resolved = {s['resolved_mismatched_orders']:,} -> {flag}")
    for rule in s.get("failed_rules", []):
        print(f"   FAIL {rule}")


print(f"Order to corrupt: {ORDER_KEY}  (+${AMOUNT} on o_totalprice)")
display(spark.table(ORDERS).filter(F.col("o_orderkey") == ORDER_KEY).select("o_orderkey", "o_orderdate", "o_totalprice"))

In [ ]:
version_before = spark.sql(f"DESCRIBE HISTORY {ORDERS} LIMIT 1").first()["version"]
print(f"silver.orders is at Delta version {version_before}\n")

try:
    # 2. corrupt one order
    spark.sql(f"UPDATE {ORDERS} SET o_totalprice = o_totalprice + {AMOUNT} WHERE o_orderkey = {ORDER_KEY}")
    print(f"Step 1 - corrupted order {ORDER_KEY}")

    # 3. the validation catches it and the alert fires
    corrupted = run_validation()
    show("Step 2 - validation on corrupted data", corrupted)
    assert corrupted["alert"] and ORDER_KEY in corrupted["new_order_keys_sample"], "The alert did not fire!"
    display(spark.table(f"{MON}.mismatched_orders_latest")
            .select("order_key", "recorded_total", "billed_total", "diff", "line_count", "mismatch_reason"))
finally:
    # 4. put silver back exactly as it was
    spark.sql(f"RESTORE TABLE {ORDERS} TO VERSION AS OF {version_before}")
    print(f"\nStep 3 - restored {ORDERS} to version {version_before}")

restored = run_validation()
show("Step 4 - validation after restore", restored)
assert restored["mismatched_orders"] == 0, "Silver still has mismatched orders after the restore"

In [ ]:
display(
    spark.table(f"{MON}.metric_mismatched_orders")
    .orderBy(F.desc("run_ts")).limit(5)
    .select("run_ts", "run_label", "mismatched_orders", "new_mismatched_orders", "resolved_mismatched_orders", "alert")
)